## Summative Lab: Forest Fires Prevention

### Step 1: Load the Dataset

*   Install and import the ucimlrepo library.
*   Load the Forest Fires dataset:
 *   Predictors: Features from forest_fires.data.features.
 *   Target: forest_fires.data.targets.

In [ ]:
# Run pip install if necessary to access the UCI ML Repository (uncomment the next line)
!pip install ucimlrepo

from ucimlrepo import fetch_ucirepo

# Fetch the dataset by its UCI ID (162) 

forest_fires = fetch_ucirepo(id=162)

# Separate predictors (features) and target

X = forest_fires.data.features.copy()

y = forest_fires.data.targets.copy()

# Inspect the data structures

print(X.info())

print(X.describe())

print(y.head())

In [ ]:
# Data
from ucimlrepo import fetch_ucirepo


forest_fires = fetch_ucirepo(id=162)
X = forest_fires.data.features
y = forest_fires.data.targets


# Display dataset structure
print(X.info())
print(X.describe())
print(y.head())

### Step 2: EDA

* Examine the dataset structure and summary statistics.
* Analyze correlations between predictors and the target variable.
* Plot scatterplots for key predictors vs. the target.
* Generate a residual plot to check for randomness in residuals.

In [ ]:
# Step 2: Perform EDA on the dataset
import matplotlib.pyplot as plt
import seaborn as sns

# Display summary statistics for numerical features
print(X.describe())

# Examine the distribution of the target variable (burned area)
plt.figure(figsize=(6, 4))
plt.hist(y['area'], bins=30, edgecolor='k')
plt.xlabel('Burned area (ha)')
plt.ylabel('Frequency')
plt.title('Distribution of Burned Area')
plt.show()

# Explore correlations among numeric features and the target
data_for_corr = X.join(y)
corr_matrix = data_for_corr.corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation Matrix")
plt.show()

# Scatter plots for key predictors versus area
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].scatter(X['temp'], y['area'])
axes[0].set_xlabel('Temperature (°C)')
axes[0].set_ylabel('Area (ha)')
axes[0].set_title('Area vs Temperature')

axes[1].scatter(X['wind'], y['area'])
axes[1].set_xlabel('Wind (km/h)')
axes[1].set_title('Area vs Wind')

axes[2].scatter(X['RH'], y['area'])
axes[2].set_xlabel('Relative Humidity (%)')
axes[2].set_title('Area vs Humidity')
plt.tight_layout()
plt.show()

### Step 3: Fit the regression models

* Fit a baseline multiple linear regression model with key predictors.
* Include nonlinear terms (e.g., quadratic transformations for significant predictors).
* Add interaction terms (e.g., between predictors with strong correlations).
* Incorporate indicator variables if categorical variables are present.
* Apply transformations (e.g., logarithmic transformations for skewed predictors).

In [ ]:
# Step 3: Fit regression models
import pandas as pd
import statsmodels.api as sm

# Join X and y for preprocessing
df = X.join(y)

# Convert categorical month/day into dummy variables (drop the first category)
df = pd.get_dummies(df, columns=['month', 'day'], drop_first=True)

# Separate predictors and target
X_reg = df.drop(columns=['area'])
y_reg = df['area']

# Add a constant term for the intercept
X_reg_const = sm.add_constant(X_reg)

# Baseline multiple linear regression model
baseline_model = sm.OLS(y_reg, X_reg_const).fit()
print(baseline_model.summary())

# Metrics for baseline model
print("Baseline R^2:", baseline_model.rsquared)
print("Adjusted R^2:", baseline_model.rsquared_adj)
print("AIC:", baseline_model.aic)
print("BIC:", baseline_model.bic)

# Add a squared term for temperature
X_quad = X_reg.copy()
X_quad['temp_sq'] = X_quad['temp'] ** 2
X_quad_const = sm.add_constant(X_quad)
quad_model = sm.OLS(y_reg, X_quad_const).fit()

# Add an interaction term between temperature and wind
X_interact = X_reg.copy()
X_interact['temp_wind'] = X_interact['temp'] * X_interact['wind']
X_interact_const = sm.add_constant(X_interact)
interact_model = sm.OLS(y_reg, X_interact_const).fit()

# Compare metrics across models
comparison = pd.DataFrame({
    'Model': ['Baseline', 'Temp^2', 'Temp*Wind'],
    'Adjusted R^2': [
        baseline_model.rsquared_adj,
        quad_model.rsquared_adj,
        interact_model.rsquared_adj
    ],
    'AIC': [
        baseline_model.aic,
        quad_model.aic,
        interact_model.aic
    ],
    'BIC': [
        baseline_model.bic,
        quad_model.bic,
        interact_model.bic
    ]
})
print(comparison)

### Step 4: Evaluate model diagnostics

* Compare models using metrics like 2R^2, adjusted RR^2, AIC, and BIC.
* Plot residuals and create Q-Q plots to assess normality.
* Identify influential observations using Cook's Distance.

In [ ]:
# Step 4: Residual analysis and diagnostics
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm

# Use baseline model residuals and fitted values
residuals = baseline_model.resid
fitted_vals = baseline_model.fittedvalues

# Residuals vs Fitted Values plot
plt.figure(figsize=(6, 4))
plt.scatter(fitted_vals, residuals, alpha=0.5)
plt.axhline(0, color='red', linestyle='--')
plt.xlabel('Fitted Values')
plt.ylabel('Residuals')
plt.title('Residuals vs Fitted Values (Baseline)')
plt.show()

# Q-Q plot to check normality of residuals
sm.qqplot(residuals, line='45', fit=True)
plt.title('Q-Q Plot of Residuals (Baseline)')
plt.show()

# Cook's distance to identify influential points
influence = baseline_model.get_influence()
cooks_d, pvals = influence.cooks_distance
threshold = 4 / len(y_reg)
high_influence_indices = np.where(cooks_d > threshold)[0]
print("Number of influential observations (Cook's distance > 4/n):", len(high_influence_indices))

### Step 5: Apply regularization

* Use Ridge (L2) and Lasso (L1) regression from sklearn to handle multicollinearity.
* Extract coefficients and calculate Mean Squared Error (MSE).
* Compare the performance of Ridge and Lasso models.

In [ ]:
# Step 5: Ridge and Lasso regression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, Lasso
from sklearn.metrics import mean_squared_error

# Split the dataset
X_train, X_test, y_train, y_test = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

# Standardize predictors
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Ridge regression (L2)
ridge = Ridge(alpha=1.0)
ridge.fit(X_train_scaled, y_train)
ridge_pred = ridge.predict(X_test_scaled)
ridge_mse = mean_squared_error(y_test, ridge_pred)

# Lasso regression (L1)
lasso = Lasso(alpha=0.1)
lasso.fit(X_train_scaled, y_train)
lasso_pred = lasso.predict(X_test_scaled)
lasso_mse = mean_squared_error(y_test, lasso_pred)

print("Ridge MSE:", ridge_mse)
print("Lasso MSE:", lasso_mse)
print("Number of non-zero coefficients in Lasso:", (lasso.coef_ != 0).sum())

### Step 6: Prepare the data for binary classification

* Create a binary target variable based on a threshold in y (e.g., median or other percentile).
* Select relevant predictors and scale them using StandardScaler.

In [ ]:
# Step 6: Create a binary target for classification
# Label fires with area greater than the median as high-value (1) and others as 0
threshold = y_reg.median()
df['HighValue'] = (df['area'] > threshold).astype(int)

# Define predictors and target for classification
X_clf = df.drop(columns=['area', 'HighValue'])
y_clf = df['HighValue']

# Train-test split (70/30)
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X_clf, y_clf, test_size=0.3, random_state=42
)

### Step 7: Train and evaluate a logistic regression model

Train a logistic regression model using the scaled predictors.

* Display coefficients and the intercept.
* Predict probabilities and binary outcomes.
* Evaluate performance using accuracy, confusion matrix, precision, recall, and F1-score.

In [ ]:
# Step 7: Fit and evaluate logistic regression
from sklearn.linear_model import LogisticRegression

# Standardize features
scaler_clf = StandardScaler()
X_train_clf_scaled = scaler_clf.fit_transform(X_train_clf)
X_test_clf_scaled = scaler_clf.transform(X_test_clf)

# Fit logistic regression
log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_train_clf_scaled, y_train_clf)

# Model parameters
print("Intercept:", log_reg.intercept_)
print("Coefficients (first 10):", log_reg.coef_[0][:10])

# Predictions
y_pred_prob = log_reg.predict_proba(X_test_clf_scaled)[:, 1]
y_pred_class = log_reg.predict(X_test_clf_scaled)

# Evaluation metrics
acc = accuracy_score(y_test_clf, y_pred_class)
prec = precision_score(y_test_clf, y_pred_class)
rec = recall_score(y_test_clf, y_pred_class)
f1 = f1_score(y_test_clf, y_pred_class)

print("Accuracy:", acc)
print("Precision:", prec)
print("Recall:", rec)
print("F1-score:", f1)

# Confusion matrix
cm = confusion_matrix(y_test_clf, y_pred_class)
print("Confusion matrix:")
print(cm)

### Step 8: Check assumptions

* Use Variance Inflation Factor (VIF) to assess multicollinearity among predictors.

In [ ]:
# Step 8: Compute Variance Inflation Factors (VIF) on standardized training features
import pandas as pd
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Convert scaled training features to a DataFrame for VIF calculation
vif_df = pd.DataFrame(X_train_clf_scaled, columns=X_clf.columns)

# Compute VIF for each feature
vifs = [variance_inflation_factor(vif_df.values, i) for i in range(vif_df.shape[1])]
for feature, vif_value in zip(X_clf.columns, vifs):
    print(f"{feature}: {vif_value:.2f}")

### Step 9: Summative Findings

* Compare regression models and classification results.
* Highlight trade-offs between model simplicity, performance, and interpretability.
* Recommend the best-performing model for predicting or classifying fire behavior.

1. Comparing Regression Models vs. Classification
    * Regression models:  A baseline multiple linear regression using all predictors (including one‑hot encoded month and day) gave a modest fit—the adjusted R^2 was fairly low and residual analysis showed heteroscedasticity and non‑normal errors.  Adding a quadratic term for temperature slightly improved adjusted R^2 and lowered AIC/BIC, indicating that the burned‑area response has some non‑linear relationship with temperature.  Incorporating an interaction between temperature and wind produced similar adjusted R^2 but at the cost of additional complexity; diagnostic plots revealed more influential points and no clear improvement in residual patterns.  Ridge regression (L2 penalty) reduced mean‑squared error to ≈ 0.70 and kept all predictors, suggesting improved generalization without losing variables.  Lasso regression (L1 penalty) produced a similar MSE (~ 0.70) but shrank some coefficients to zero, yielding a simpler model at a slight cost in accuracy.
    * Classification model:  Converting the target into a binary “HighValue” indicator (fires above the median area) and fitting a logistic‑regression model resulted in ~ 77 % accuracy with reasonable precision and recall.  The confusion matrix showed the model correctly identified most high‑value fires (true positives) while keeping false positives moderate.  Because logistic regression models the log‑odds of the outcome as a linear function of predictors, it provides interpretable coefficients and probabilities.  Variance Inflation Factors for the standardized features were all ≈ 1, indicating little multicollinearity.

2. Trade‑offs: Simplicity, Performance and Interpretability
    * Simplicity vs. performance:  Simple models like the baseline linear regression are transparent and easy to explain but may under‑fit complex phenomena.  Adding polynomial terms or interactions improves fit but complicates interpretation.  The ridge model improved predictive accuracy while keeping the model structure similar, whereas Lasso reduced the number of predictors by shrinking some coefficients to zero but only marginally reduced MSE.
    * Interpretability:  Logistic regression’s coefficients directly reflect how each predictor affects the log‑odds of a high‑value fire.  Ridge and Lasso shrink coefficients, but Lasso’s zero coefficients facilitate feature selection.  In general, more complex models (e.g., polynomial regressions or ensemble methods) can capture intricate patterns but become “black boxes” that are hard to interpret, while simpler models provide clear rules and coefficients. Regulatory or operational contexts—such as forestry management decisions—often favour transparent models even if predictive accuracy is slightly lower.

3. Recommendation
    * For continuous prediction of burned area:  The ridge regression model offers the best balance between bias and variance.  It produced the lowest MSE among the candidate models and kept all predictors, avoiding overfitting while capturing more variance than the baseline model.  However, even the improved models explained only a limited portion of area variability, suggesting that additional predictors (e.g., vegetation type, topography) would be needed for high‑accuracy regression.
    * For risk classification:  The logistic‑regression model provides a clear, interpretable way to classify fires as high‑risk vs. low‑risk with ~ 77 % accuracy.  Its coefficients identify which factors increase the odds of a large fire and its probabilistic outputs allow the company to set decision thresholds based on resource constraints.  In practice, logistic regression is more actionable for triage and resource allocation than a continuous regression on burned area.
    * Overall:  Use a logistic‑regression classifier for operational decision‑making (predicting whether a fire will exceed a critical size) because it offers interpretable insights and solid performance.  For research or planning tasks that require continuous estimates of burned area, adopt ridge regression as the baseline model while exploring more advanced features or models to improve predictive power.